In [1]:
from pyetc_wst import WST
from pyetc_wst.etc import snr_in_window
import numpy as np
wst = WST(log = 'DEBUG', skip_dataload = False)
wst.info()

[DEBUG] WST.__init__ processing time: 12.8930 seconds
[INFO] ETC version 1.5 release date 22 June 2026
[INFO] 	- Fixed bug in _resolve_best_coadd_ifs (IFS COADD_XY="best" mode): replaced the sky-dominated approximation metric fsq/N with the correct full SNR metric signal/sqrt(signal + N²·bg_per_spaxel), where signal = fsq·S and bg_per_spaxel = sky + dark + RON per spaxel at the reference wavelength. The source spectrum is now passed from all three callers (snr_from_source_ifs, _snr_at_wave_ifs, time_from_source_ifs); when no spectrum is available the old approximation is used as fallback.
[INFO] 	- Increased default max_coadd from 20 to 40 for point sources: bad seeing conditions (e.g. 1.5–2") with small spaxels can push the optimal aperture close to or beyond the old cap.
[INFO] 	- For resolved sources, max_coadd is now derived automatically from the extent of the source image (min(ima.shape) // oversamp), removing the artificial fixed ceiling and ensuring the full extent of extended 

In [13]:
def wst_ect(tracer, detection_band,channel, exposure_time, Lam_Ref, max_mag, fli, obj_sed, redshift):

    temp = 'uniform' if obj_sed == 'uniform' else 'template'
    
    base_obs = {"INS": 'moslr',
                "CH": channel,
                "NDIT": 1,
                "DIT": exposure_time,#1000,
                "SNR": 0.,
                "Lam_Ref": Lam_Ref,
                "OBJ_FIB_DISP": 0,
                "SEE": 1.1,
                "GLAO": False,
                "SKYCALC": True,
                "FLI": fli,
                "MOON_SEP": 45,
                "AM": 1.2,
                "PMW": 5,
                "Obj_SED": temp,
                "SED_Name": obj_sed,
                "OBJ_MAG": max_mag,
                "MAG_SYS": "AB",
                "MAG_FIL": detection_band+"SDSS",
                "Z": redshift,
                "BB_Temp": 9000.0,
                "PL_Index": -2,
                "SEL_FLUX": None,
                "SEL_CWAV": None,
                "SEL_FWHM": None,
                "Obj_Spat_Dis": "ps",
                "IMA": 'sersic',
                "IMA_FWHM": 1.1,
                "IMA_BETA": None,
                "Sersic_Reff": 0.3,
                "Sersic_Ind": 1,
                "COADD_WL": 1,
                "COADD_XY": 1,
                "SNR_RANGE": False,
                "LAM_WIN1": 4000,
                "LAM_WIN2": 10000}
    
    con, ob, spe, im, spe_input = wst.build_obs_full(base_obs)
    return con, ob, spe, im, spe_input


In [64]:
tracer = ['BG', 'LRG', 'ELG', 'LBGu', 'LBGg', 'LBGr']
detection_band = ['r', 'z', 'g', 'r', 'i', 'z']
fli = ['0.8', '0.5', '0.5', '0.', '0.', '0.']
obj_sed = ['Kinney_ell','Kinney_ell', 'Kinney_starb2', 'uniform', 'uniform', 'uniform']
redshift = [0.3, 0.8, 1.2, 0, 0, 0]
max_mag_desi = [19.5, 21.5, 24.1, 24.2, 24.2, 24.2]
max_mag_wst = [21.7, 22.8, 25.2, 25.1, 24.6, 25]
desi_exposure_time = [180, 1000, 1000, 2*60*60, 0, 0]
Sdesi = 3.8**2
Swst = 12**2
desi_to_wst_exposure_time = np.array(desi_exposure_time) * Sdesi/Swst
channel = ["yellow",  "red", "green", "yellow", "yellow", "red"]
Lam_Ref = 0*np.array([7000, 6500, 5500, 7000, 7000, 6500])

In [67]:
for i in range(6):
   print(tracer[i])
   print('detection band:', detection_band[i])
   #print('MOS-LR channel:', channel[i])
   print(f'base DESI exposure time: {desi_exposure_time[i]/60:.2f} minutes')
   print('base DESI maximum magnitude:', max_mag_desi[i],)
   con, ob, spe, im, spe_input = wst_ect(tracer[i], detection_band[i], channel[i], desi_to_wst_exposure_time[i], Lam_Ref[i], max_mag_desi[i], fli[i], obj_sed[i], redshift[i])

   res_snr = wst.snr_from_source(con, im, spe, debug=False)
   med = snr_in_window(res_snr, lam1=100, lam2=200000)
   if tracer[i] == 'BG': med = 1
   if tracer[i] == 'ELG': med = 0.1
   if tracer[i] == 'LBGg': med = 0.4
  # if tracer[i] == 'LBGu': med = 0.35
   elif tracer[i] == 'LBGr': med = 0.13
    
   print(f'target observed {desi_exposure_time[i]/60:.2f} minutes by DESI (or {desi_to_wst_exposure_time[i]/60:.2f} minutes by WST) => SNR median in channel {channel[i]} = {med:.3f}')

   con, ob, spe, im, spe_input = wst_ect(tracer[i], detection_band[i], channel[i], desi_to_wst_exposure_time[i], Lam_Ref[i], max_mag_wst[i], fli[i], obj_sed[i], redshift[i])

   result = wst.time_from_source_window(con, im, spe,
                                         lam1=100, lam2=200000,
                                         target_snr=med,
                                         compute='dit', debug=False)
   print(f'keep SNR median in channel {channel[i]} = {result['median_snr']:.3f} for "new" {detection_band[i]}-max = {max_mag_wst[i]:.2f} => {result['dit']/60:2f} minutes by WST')

   print()
   #break

BG
detection band: r
base DESI exposure time: 3.00 minutes
base DESI maximum magnitude: 19.5
target observed 3.00 minutes by DESI (or 0.30 minutes by WST) => SNR median in channel yellow = 1.000
keep SNR median in channel yellow = 1.000 for "new" r-max = 21.70 => 4.119237 minutes by WST

LRG
detection band: z
base DESI exposure time: 16.67 minutes
base DESI maximum magnitude: 21.5
target observed 16.67 minutes by DESI (or 1.67 minutes by WST) => SNR median in channel red = 0.237
keep SNR median in channel red = 0.237 for "new" z-max = 22.80 => 13.536536 minutes by WST

ELG
detection band: g
base DESI exposure time: 16.67 minutes
base DESI maximum magnitude: 24.1
target observed 16.67 minutes by DESI (or 1.67 minutes by WST) => SNR median in channel green = 0.100
keep SNR median in channel green = 0.100 for "new" g-max = 25.20 => 20.631769 minutes by WST

LBGu
detection band: r
base DESI exposure time: 120.00 minutes
base DESI maximum magnitude: 24.2
target observed 120.00 minutes by DE